In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
PANEL_PATH = Path("parquet_output/h3_month_panel.parquet")

panel = pd.read_parquet(PANEL_PATH)

print(panel.shape)
display(panel.head())
display(panel.dtypes)

In [ ]:
target_col = "target_had_accident_next_month"

print("Target distribution:")
display(panel[target_col].value_counts(dropna=False))
display(panel[target_col].value_counts(normalize=True, dropna=False))

print("\nMonth range:")
print(panel["month"].min(), "—", panel["month"].max())

print("\nRows by year:")
display(panel["year"].value_counts().sort_index())

In [ ]:
monthly_target_rate = (
    panel.groupby("month")[target_col]
    .mean()
    .reset_index(name="target_rate")
)

display(monthly_target_rate.head())
display(monthly_target_rate.tail())

monthly_target_rate.plot(
    x="month",
    y="target_rate",
    figsize=(14, 4),
    title="Target rate by month",
)

In [ ]:
features = [
    "year",
    "month_num",
    "accidents_prev_1m",
    "accidents_prev_3m",
    "accidents_prev_6m",
    "accidents_prev_12m",
]

target = "target_had_accident_next_month"

df = panel[["h3", "month", *features, target]].copy()

df["month"] = pd.to_datetime(df["month"])
df[target] = df[target].astype(int)

display(df.head())

In [ ]:
TRAIN_END = "2023-01-01"

train = df[df["month"] < TRAIN_END].copy()
test = df[df["month"] >= TRAIN_END].copy()

print("Train:", train.shape, train["month"].min(), train["month"].max())
print("Test:", test.shape, test["month"].min(), test["month"].max())

print("\nTrain target rate:", train[target].mean())
print("Test target rate:", test[target].mean())

In [ ]:
X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [ ]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        n_jobs=None,
    )),
])

logreg.fit(X_train, y_train)

test_scores = logreg.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= 0.5).astype(int)

print("Done")

In [ ]:
roc_auc = roc_auc_score(y_test, test_scores)
pr_auc = average_precision_score(y_test, test_scores)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"Base positive rate in test: {y_test.mean():.4f}")

In [ ]:
print("Classification report with threshold=0.5")
print(classification_report(y_test, test_pred, digits=4))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

In [ ]:
def precision_recall_at_k(y_true, scores, k_frac=None, k_abs=None):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)

    n = len(y_true)

    if k_abs is None:
        if k_frac is None:
            raise ValueError("Pass k_frac or k_abs")
        k_abs = max(1, int(n * k_frac))

    k_abs = min(k_abs, n)

    top_idx = np.argsort(scores)[::-1][:k_abs]

    tp_at_k = y_true[top_idx].sum()
    positives = y_true.sum()

    precision = tp_at_k / k_abs
    recall = tp_at_k / positives if positives > 0 else np.nan
    lift = precision / y_true.mean() if y_true.mean() > 0 else np.nan

    return {
        "k": k_abs,
        "precision": precision,
        "recall": recall,
        "lift": lift,
        "positives_in_top_k": int(tp_at_k),
        "total_positives": int(positives),
    }


for k_frac in [0.01, 0.03, 0.05, 0.10]:
    m = precision_recall_at_k(y_test, test_scores, k_frac=k_frac)
    print(
        f"Top {k_frac:.0%}: "
        f"k={m['k']:,}, "
        f"precision={m['precision']:.4f}, "
        f"recall={m['recall']:.4f}, "
        f"lift={m['lift']:.2f}, "
        f"tp={m['positives_in_top_k']:,}/{m['total_positives']:,}"
    )

In [ ]:
test_scored = test[["h3", "month", target]].copy()
test_scored["score"] = test_scores

display(test_scored.head())

In [ ]:
def monthly_precision_recall_at_k(df, target_col, score_col, k_frac=0.05):
    rows = []

    for month, g in df.groupby("month"):
        y_true = g[target_col].to_numpy().astype(int)
        scores = g[score_col].to_numpy()

        if len(g) == 0:
            continue

        metrics = precision_recall_at_k(y_true, scores, k_frac=k_frac)
        metrics["month"] = month
        metrics["n_rows"] = len(g)
        metrics["positive_rate"] = y_true.mean()
        rows.append(metrics)

    return pd.DataFrame(rows)


monthly_metrics = monthly_precision_recall_at_k(
    test_scored,
    target_col=target,
    score_col="score",
    k_frac=0.05,
)

display(monthly_metrics)
display(monthly_metrics.describe())

In [ ]:
print("Mean monthly metrics @ top 5%:")
print("precision:", monthly_metrics["precision"].mean())
print("recall:", monthly_metrics["recall"].mean())
print("lift:", monthly_metrics["lift"].mean())